# T2 - Pipeline Final: MediaPipe Landmarks + SVM

## Objetivo

Este notebook apresenta a melhor estratégia encontrada para a continuação do T1: classificar poses de yoga usando **landmarks corporais** extraídos por uma rede neural pré-treinada e um classificador SVM.

A pipeline final é:

Imagem de entrada
→ MediaPipe Pose
→ Extração de 33 landmarks corporais
→ Normalização geométrica da pose
→ Treinamento de SVM
→ Avaliação com métricas e matriz de confusão

O ponto principal é que a rede neural não classifica a imagem inteira. Ela é usada para detectar a estrutura corporal. Depois disso, o SVM classifica a pose a partir da geometria do corpo.

## 0. Dependências

Este notebook deve ser executado no ambiente MediaPipe:

```bash
pip install -r requirements-mediapipe.txt
python -m ipykernel install --user --name yoga-mediapipe --display-name "Python (yoga-mediapipe)"
```

No Jupyter, selecione o kernel **Python (yoga-mediapipe)**.

In [ ]:
from pathlib import Path
import random

import cv2
import joblib
import mediapipe as mp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from tqdm import tqdm

if not hasattr(mp, "solutions"):
    raise RuntimeError(
        "A versão instalada do MediaPipe não possui mp.solutions.pose. "
        "Use o ambiente correto e instale: pip install -r requirements-mediapipe.txt"
    )

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DATASET_DIR = Path("dataset")
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
SVM_MODEL_PATH = MODEL_DIR / "t2_landmarks_svm.joblib"
METADATA_PATH = MODEL_DIR / "t2_landmarks_metadata.joblib"

assert DATASET_DIR.exists(), f"Dataset não encontrado: {DATASET_DIR.resolve()}"

print("Dataset:", DATASET_DIR.resolve())
print("MediaPipe:", getattr(mp, "__version__", "sem versão"))

## 1. Leitura do dataset

As imagens estão organizadas por pastas, em que cada pasta representa uma classe. Pastas vazias são ignoradas.

In [ ]:
def load_image_table(dataset_dir, extensions):
    samples = []

    for class_dir in sorted(dataset_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        image_paths = sorted(
            path for path in class_dir.rglob("*")
            if path.is_file() and path.suffix.lower() in extensions
        )

        for image_path in image_paths:
            samples.append({
                "path": str(image_path),
                "label": class_dir.name,
            })

    image_df = pd.DataFrame(samples)
    assert len(image_df) > 0, "Nenhuma imagem válida encontrada."
    return image_df


image_df = load_image_table(DATASET_DIR, IMG_EXTENSIONS)
class_names = sorted(image_df["label"].unique())
class_to_id = {class_name: index for index, class_name in enumerate(class_names)}
id_to_class = {index: class_name for class_name, index in class_to_id.items()}

image_df["label_id"] = image_df["label"].map(class_to_id).astype("int32")

print("Total de imagens:", len(image_df))
print("Total de classes:", len(class_names))
display(image_df.head())

In [ ]:
class_summary = (
    image_df.groupby("label")
    .size()
    .reset_index(name="image_count")
    .sort_values("image_count", ascending=False)
    .reset_index(drop=True)
)

display(class_summary)

plt.figure(figsize=(12, 6))
sns.histplot(class_summary["image_count"], bins=20)
plt.title("Distribuição de imagens por classe")
plt.xlabel("Imagens por classe")
plt.ylabel("Quantidade de classes")
plt.tight_layout()
plt.show()

## 2. Separação em treino, validação e teste

Usamos validação para escolher o melhor valor de `C` do SVM e teste para a avaliação final.

In [ ]:
train_df, temp_df = train_test_split(
    image_df,
    test_size=VALIDATION_SIZE + TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=image_df["label_id"],
)

test_size_adjusted = TEST_SIZE / (VALIDATION_SIZE + TEST_SIZE)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=test_size_adjusted,
    random_state=RANDOM_STATE,
    stratify=temp_df["label_id"],
)

print("Treino:", train_df.shape)
print("Validação:", validation_df.shape)
print("Teste:", test_df.shape)

split_summary = pd.DataFrame({
    "split": ["treino", "validacao", "teste"],
    "images": [len(train_df), len(validation_df), len(test_df)],
    "percentage": [len(train_df) / len(image_df), len(validation_df) / len(image_df), len(test_df) / len(image_df)],
})

split_summary

## 3. Extração dos landmarks

O MediaPipe Pose retorna 33 pontos corporais. Cada ponto possui `x`, `y`, `z` e `visibility`.

Antes de treinar o SVM, normalizamos os pontos:

- centralizamos a pose no quadril;
- dividimos pelo tamanho do torso/ombros/quadril;
- reduzimos o efeito de escala e posição da pessoa na imagem.

In [ ]:
POSE_LANDMARKS = 33
LANDMARK_DIMS = 4
LANDMARK_FEATURES = POSE_LANDMARKS * LANDMARK_DIMS

LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12
LEFT_HIP = 23
RIGHT_HIP = 24


def normalize_landmarks(landmarks):
    points = landmarks.astype("float32").copy()

    hip_center = (points[LEFT_HIP, :2] + points[RIGHT_HIP, :2]) / 2.0
    shoulder_center = (points[LEFT_SHOULDER, :2] + points[RIGHT_SHOULDER, :2]) / 2.0

    shoulder_width = np.linalg.norm(points[LEFT_SHOULDER, :2] - points[RIGHT_SHOULDER, :2])
    hip_width = np.linalg.norm(points[LEFT_HIP, :2] - points[RIGHT_HIP, :2])
    torso_size = np.linalg.norm(shoulder_center - hip_center)
    scale = max(shoulder_width, hip_width, torso_size, 1e-6)

    points[:, 0] = (points[:, 0] - hip_center[0]) / scale
    points[:, 1] = (points[:, 1] - hip_center[1]) / scale
    points[:, 2] = points[:, 2] / scale

    return points.reshape(-1).astype("float32")


def extract_landmarks_from_image(image_path, pose):
    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        return None

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    result = pose.process(image_rgb)

    if not result.pose_landmarks:
        return None

    landmarks = np.array(
        [[point.x, point.y, point.z, point.visibility] for point in result.pose_landmarks.landmark],
        dtype="float32",
    )

    if landmarks.shape != (POSE_LANDMARKS, LANDMARK_DIMS):
        return None

    return normalize_landmarks(landmarks)

In [ ]:
def extract_landmarks_dataset(split_df, split_name):
    features = []
    labels = []
    valid_rows = []
    failed_paths = []

    mp_pose = mp.solutions.pose

    with mp_pose.Pose(
        static_image_mode=True,
        model_complexity=2,
        enable_segmentation=False,
        min_detection_confidence=0.30,
    ) as pose:
        for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"Landmarks {split_name}"):
            landmark_vector = extract_landmarks_from_image(row["path"], pose)

            if landmark_vector is None:
                failed_paths.append(row["path"])
                continue

            features.append(landmark_vector)
            labels.append(int(row["label_id"]))
            valid_rows.append(row)

    features = np.vstack(features).astype("float32")
    labels = np.array(labels, dtype="int32")
    valid_df = pd.DataFrame(valid_rows).reset_index(drop=True)

    print(f"{split_name}: {len(valid_df)} imagens com pose detectada / {len(split_df)}")
    print(f"{split_name}: {len(failed_paths)} imagens sem pose detectada")

    return features, labels, valid_df, failed_paths


X_train, y_train, train_valid_df, train_failed_paths = extract_landmarks_dataset(train_df, "treino")
X_val, y_val, validation_valid_df, validation_failed_paths = extract_landmarks_dataset(validation_df, "validação")
X_test, y_test, test_valid_df, test_failed_paths = extract_landmarks_dataset(test_df, "teste")

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

In [ ]:
detection_summary = pd.DataFrame([
    {"split": "treino", "detected_images": len(train_valid_df), "total_images": len(train_df)},
    {"split": "validacao", "detected_images": len(validation_valid_df), "total_images": len(validation_df)},
    {"split": "teste", "detected_images": len(test_valid_df), "total_images": len(test_df)},
])

detection_summary["detection_rate"] = detection_summary["detected_images"] / detection_summary["total_images"]
display(detection_summary)

## 4. Treinamento do SVM

Treinamos um SVM com kernel RBF. O `StandardScaler` padroniza os landmarks antes da classificação.

O melhor valor de `C` é escolhido no conjunto de validação.

In [ ]:
validation_results = []
best_model = None
best_accuracy = -1
best_C = None

for C in [0.1, 1, 3, 10, 30, 100]:
    svm_model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=C, gamma="scale", class_weight="balanced", probability=True)),
    ])

    svm_model.fit(X_train, y_train)
    y_val_pred = svm_model.predict(X_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)

    validation_results.append({"C": C, "validation_accuracy": val_accuracy})

    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_C = C
        best_model = svm_model


df_validation = pd.DataFrame(validation_results)
display(df_validation)

print("Melhor C:", best_C)
print(f"Melhor acurácia na validação: {best_accuracy:.4f}")

## 5. Avaliação final

O teste é usado apenas no final, depois da escolha do melhor `C`.

In [ ]:
y_pred = best_model.predict(X_test)
y_confidence = best_model.predict_proba(X_test).max(axis=1)

accuracy_landmarks_svm = accuracy_score(y_test, y_pred)

print(f"Acurácia no teste com MediaPipe landmarks + SVM: {accuracy_landmarks_svm:.4f}")
print(f"Acurácia percentual: {accuracy_landmarks_svm * 100:.2f}%")
print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    zero_division=0,
))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(class_names)))

plt.figure(figsize=(16, 14))
sns.heatmap(
    cm,
    cmap="Greens",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=True,
)
plt.title("Matriz de Confusão - MediaPipe landmarks + SVM")
plt.xlabel("Classe predita")
plt.ylabel("Classe real")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Análise dos erros

Esta tabela ajuda a explicar quais poses ainda são confundidas.

In [ ]:
results_df = test_valid_df.copy().reset_index(drop=True)
results_df["true_id"] = y_test
results_df["pred_id"] = y_pred
results_df["true_label"] = [id_to_class[int(label)] for label in y_test]
results_df["pred_label"] = [id_to_class[int(label)] for label in y_pred]
results_df["confidence"] = y_confidence

errors_df = results_df[results_df["true_id"] != results_df["pred_id"]]

print("Total de amostras de teste com pose detectada:", len(results_df))
print("Total de erros:", len(errors_df))
print(f"Taxa de erro: {len(errors_df) / len(results_df):.4f}")

display(results_df.head())

In [ ]:
confused_pairs = (
    errors_df.groupby(["true_label", "pred_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(15)
)

display(confused_pairs)

In [ ]:
def show_error_examples(errors_df, n=12):
    if len(errors_df) == 0:
        print("Nenhum erro encontrado.")
        return

    samples = errors_df.sample(n=min(n, len(errors_df)), random_state=RANDOM_STATE).reset_index(drop=True)
    cols = 4
    rows = int(np.ceil(len(samples) / cols))

    plt.figure(figsize=(14, 4 * rows))

    for i, row in samples.iterrows():
        image_bgr = cv2.imread(row["path"])
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        plt.subplot(rows, cols, i + 1)
        plt.imshow(image_rgb)
        plt.title(
            f"Real: {row['true_label']}\nPred: {row['pred_label']}\nConf: {row['confidence']:.2f}",
            fontsize=9,
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()


show_error_examples(errors_df)

## 7. Comparação com o T1

Resultados de referência do T1:

- Pixels crus + SVM: **47,46%**
- HOG + SVM: **69,57%**

O resultado deste notebook mostra o ganho ao trocar a representação da imagem por landmarks corporais.

In [ ]:
comparison_df = pd.DataFrame([
    {"experiment": "Pixels crus + SVM (T1)", "accuracy": 0.4746},
    {"experiment": "HOG + SVM (T1)", "accuracy": 0.6957},
    {"experiment": "MediaPipe landmarks + SVM (T2)", "accuracy": accuracy_landmarks_svm},
])

comparison_df["accuracy_percent"] = comparison_df["accuracy"] * 100
display(comparison_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=comparison_df, x="experiment", y="accuracy_percent")
plt.title("Comparação entre pipelines")
plt.ylabel("Acurácia (%)")
plt.xlabel("")
plt.ylim(0, 100)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 8. Salvando o modelo

O modelo final e os metadados das classes são salvos para permitir demo posterior.

In [ ]:
joblib.dump(best_model, SVM_MODEL_PATH)
joblib.dump(
    {
        "class_names": class_names,
        "class_to_id": class_to_id,
        "id_to_class": id_to_class,
        "best_C": best_C,
        "test_accuracy": accuracy_landmarks_svm,
        "landmark_features": LANDMARK_FEATURES,
    },
    METADATA_PATH,
)

print("Modelo salvo em:", SVM_MODEL_PATH)
print("Metadados salvos em:", METADATA_PATH)

## 9. Demo com uma imagem

A função abaixo recebe uma imagem, extrai landmarks e mostra as classes mais prováveis.

In [ ]:
def predict_image(image_path, top_k=5):
    mp_pose = mp.solutions.pose

    with mp_pose.Pose(
        static_image_mode=True,
        model_complexity=2,
        enable_segmentation=False,
        min_detection_confidence=0.30,
    ) as pose:
        landmark_vector = extract_landmarks_from_image(image_path, pose)

    if landmark_vector is None:
        print("Não foi possível detectar pose na imagem.")
        return None

    probabilities = best_model.predict_proba(landmark_vector.reshape(1, -1))[0]
    top_indices = np.argsort(probabilities)[::-1][:top_k]

    prediction_df = pd.DataFrame({
        "class_name": [id_to_class[int(index)] for index in top_indices],
        "probability": [float(probabilities[index]) for index in top_indices],
    })

    image_bgr = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(5, 5))
    plt.imshow(image_rgb)
    plt.title(f"Predição: {prediction_df.iloc[0]['class_name']}")
    plt.axis("off")
    plt.show()

    display(prediction_df)
    return prediction_df


demo_sample = test_valid_df.sample(n=1, random_state=RANDOM_STATE).iloc[0]
print("Imagem:", demo_sample["path"])
print("Classe real:", demo_sample["label"])
predict_image(demo_sample["path"])